# Character Error Rate (CER) Evaluation

In [ ]:
import re

In [ ]:
def normalize_text(text, case_sensitive=False, keep_chars=None):    if not case_sensitive:        text = text.lower()    if keep_chars is not None:        pattern = f"[^{re.escape(keep_chars)}]"        text = re.sub(pattern, "", text)    text = re.sub(r"\s+", " ", text).strip()    return list(text)

In [ ]:
def edit_distance_ops(ref_chars, hyp_chars):    n, m = len(ref_chars), len(hyp_chars)    dp = [[0] * (m + 1) for _ in range(n + 1)]    for i in range(n + 1):        dp[i][0] = i    for j in range(m + 1):        dp[0][j] = j    for i in range(1, n + 1):        for j in range(1, m + 1):            if ref_chars[i - 1] == hyp_chars[j - 1]:                dp[i][j] = dp[i - 1][j - 1]            else:                substitution = dp[i - 1][j - 1] + 1                insertion = dp[i][j - 1] + 1                deletion = dp[i - 1][j] + 1                dp[i][j] = min(substitution, insertion, deletion)    i, j = n, m    substitutions = insertions = deletions = 0    while i > 0 or j > 0:        if i > 0 and j > 0 and ref_chars[i - 1] == hyp_chars[j - 1]:            i -= 1            j -= 1        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:            substitutions += 1            i -= 1            j -= 1        elif j > 0 and dp[i][j] == dp[i][j - 1] + 1:            insertions += 1            j -= 1        else:            deletions += 1            i -= 1    return substitutions, deletions, insertions, dp[n][m]

In [ ]:
def cer(reference, hypothesis, case_sensitive=False, keep_chars=None):    ref_chars = normalize_text(reference, case_sensitive, keep_chars)    hyp_chars = normalize_text(hypothesis, case_sensitive, keep_chars)    substitutions, deletions, insertions, _ = edit_distance_ops(ref_chars, hyp_chars)    n = len(ref_chars) if len(ref_chars) > 0 else 1    return (substitutions + deletions + insertions) / n

In [ ]:
def batch_cer(pairs, case_sensitive=False, keep_chars=None):    total_s = total_d = total_i = total_n = 0    per_example = []    for reference, hypothesis in pairs:        ref_chars = normalize_text(reference, case_sensitive, keep_chars)        hyp_chars = normalize_text(hypothesis, case_sensitive, keep_chars)        s, d, i, _ = edit_distance_ops(ref_chars, hyp_chars)        n = len(ref_chars) if len(ref_chars) > 0 else 1        example_cer = (s + d + i) / n        per_example.append(example_cer)        total_s += s        total_d += d        total_i += i        total_n += len(ref_chars)    corpus_n = total_n if total_n > 0 else 1    corpus_cer = (total_s + total_d + total_i) / corpus_n    return {        "per_example_cer": per_example,        "corpus_cer": corpus_cer,        "substitutions": total_s,        "deletions": total_d,        "insertions": total_i,        "total_chars": total_n    }

In [ ]:
pairs = [    ("tesseract ocr converts images to text", "tesseract ocr convert images to text"),    ("hello world", "helo world"),    ("speech recognition is hard", "speech recognltion is hard"),    ("please call me tomorrow", "please call me tommorow"),]results = batch_cer(pairs)for (reference, hypothesis), example_cer in zip(pairs, results["per_example_cer"]):    print(f"ref: {reference}")    print(f"hyp: {hypothesis}")    print(f"cer: {example_cer:.3f}")    print("-" * 40)print(f"corpus cer: {results['corpus_cer']:.3f}")print(f"substitutions: {results['substitutions']}")print(f"deletions: {results['deletions']}")print(f"insertions: {results['insertions']}")print(f"total reference chars: {results['total_chars']}")